# Metadata Creation Exercise — Global Air Pollution Dataset

**Student:** Vinay Bommenahalli Gurubasappa   
**Course:** Applied Data Science & AI — Data Management  
**Week / Exercise:** Week 1 — Individual Metadata Creation Exercise  

---

## Goal
Practice creating metadata by applying what we learned in class to a publicly available dataset.

## Task
Select a publicly available dataset and create:
1. **DataCite metadata** in XML format
2. **schema.org metadata** in JSON-LD format using `@type: Dataset`

## Dataset chosen
**Global Air Pollution Dataset** — by Hasib Al Muzdadid  
Source: [https://www.kaggle.com/datasets/hasibalmuzdadid/global-air-pollution-dataset](https://www.kaggle.com/datasets/hasibalmuzdadid/global-air-pollution-dataset)

The dataset contains air quality index (AQI) values for multiple pollutants across countries and cities worldwide.

## Step 0 — Import Libraries

In [29]:
import pandas as pd
import json
import xml.etree.ElementTree as ET
import xml.dom.minidom
from pathlib import Path
from IPython.display import display, JSON, Markdown

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Step 1 — Load and Explore the Dataset

I load the dataset to understand its structure before writing metadata.  
The CSV file is expected at: `global air pollution dataset.csv`  


In [30]:
# Adjust the filename/path if yours differs
csv_path = Path("global air pollution dataset.csv")

if not csv_path.exists():
    print(f"File not found at '{csv_path}'.")
    print("Please place the downloaded CSV in the same folder as this notebook.")
else:
    df = pd.read_csv(csv_path)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")
    display(df.head())

Shape: 23463 rows × 12 columns
Columns: ['Country', 'City', 'AQI Value', 'AQI Category', 'CO AQI Value', 'CO AQI Category', 'Ozone AQI Value', 'Ozone AQI Category', 'NO2 AQI Value', 'NO2 AQI Category', 'PM2.5 AQI Value', 'PM2.5 AQI Category']


,Country,City,AQI Value,AQI Category,CO AQI Value,CO AQI Category,Ozone AQI Value,Ozone AQI Category,NO2 AQI Value,NO2 AQI Category,PM2.5 AQI Value,PM2.5 AQI Category
0,Russian Federation,Praskoveya,51,Moderate,1,Good,36,Good,0,Good,51,Moderate
1,Brazil,Presidente Dutra,41,Good,1,Good,5,Good,1,Good,41,Good
2,Italy,Priolo Gargallo,66,Moderate,1,Good,39,Good,2,Good,66,Moderate
3,Poland,Przasnysz,34,Good,1,Good,34,Good,0,Good,20,Good
4,France,Punaauia,22,Good,0,Good,22,Good,0,Good,6,Good


In [31]:
# Basic statistics — helps write a meaningful metadata description
display(df.describe(include='all').transpose())

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Country,23036,175,United States of America,2872,NaN,NaN,NaN,NaN,NaN,NaN,NaN
City,23462,23462,Marang,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AQI Value,23463.0,NaN,NaN,NaN,72.010868,56.05522,6.0,39.0,55.0,79.0,500.0
AQI Category,23463,6,Good,9936,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CO AQI Value,23463.0,NaN,NaN,NaN,1.368367,1.832064,0.0,1.0,1.0,1.0,133.0
CO AQI Category,23463,3,Good,23460,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ozone AQI Value,23463.0,NaN,NaN,NaN,35.193709,28.098723,0.0,21.0,31.0,40.0,235.0
Ozone AQI Category,23463,5,Good,21069,NaN,NaN,NaN,NaN,NaN,NaN,NaN
NO2 AQI Value,23463.0,NaN,NaN,NaN,3.063334,5.254108,0.0,0.0,1.0,4.0,91.0
NO2 AQI Category,23463,2,Good,23448,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 2 — Define Descriptive Metadata

Before writing formal metadata in XML or JSON-LD, we collect the key information  
in a plain Python dictionary. This mirrors the approach from class (notebook `02_day2_intro_to_metadata.ipynb`).

Three layers:
- **Descriptive** — what the dataset is about (title, description, keywords)
- **Structural** — how it is organised (columns, units, row meaning)
- **Administrative** — who owns it, under what licence, version, access

In [32]:
descriptive_metadata = {
    "title": "Global Air Pollution Dataset",
    "description": (
        "A global dataset containing Air Quality Index (AQI) values and "
        "concentration measurements for major air pollutants — CO, Ozone, "
        "NO2, and PM2.5 — across cities and countries worldwide. "
        "Each row represents one city-level observation."
    ),
    "keywords": "air quality, AQI, pollution, CO, Ozone, NO2, PM2.5, environment, global",
    "creator": "Hasib Al Muzdadid",
    "subject": "Environmental Science / Air Quality",
}

display(pd.DataFrame(
    {"Field": list(descriptive_metadata.keys()),
     "Value": [str(v) for v in descriptive_metadata.values()]}
))

,Field,Value
0,title,Global Air Pollution Dataset
1,description,A global dataset containing Air Quality Index ...
2,keywords,"air quality, AQI, pollution, CO, Ozone, NO2, P..."
3,creator,Hasib Al Muzdadid
4,subject,Environmental Science / Air Quality


In [33]:
structural_metadata = {
    "row_meaning": "Each row represents one city-level air quality observation.",
    "columns": {
        "Country": "Name of the country",
        "City": "Name of the city",
        "AQI Value": "Overall Air Quality Index value (dimensionless)",
        "AQI Category": "Categorical AQI label (Good / Moderate / Unhealthy etc.)",
        "CO AQI Value": "AQI sub-index for Carbon Monoxide",
        "CO AQI Category": "Categorical label for CO AQI",
        "Ozone AQI Value": "AQI sub-index for Ozone",
        "Ozone AQI Category": "Categorical label for Ozone AQI",
        "NO2 AQI Value": "AQI sub-index for Nitrogen Dioxide",
        "NO2 AQI Category": "Categorical label for NO2 AQI",
        "PM2.5 AQI Value": "AQI sub-index for fine particulate matter (PM2.5)",
        "PM2.5 AQI Category": "Categorical label for PM2.5 AQI",
    }
}

struct_rows = [{"Column": k, "Meaning": v} for k, v in structural_metadata["columns"].items()]
display(pd.DataFrame(struct_rows))

,Column,Meaning
0,Country,Name of the country
1,City,Name of the city
2,AQI Value,Overall Air Quality Index value (dimensionless)
3,AQI Category,Categorical AQI label (Good / Moderate / Unhea...
4,CO AQI Value,AQI sub-index for Carbon Monoxide
5,CO AQI Category,Categorical label for CO AQI
6,Ozone AQI Value,AQI sub-index for Ozone
7,Ozone AQI Category,Categorical label for Ozone AQI
8,NO2 AQI Value,AQI sub-index for Nitrogen Dioxide
9,NO2 AQI Category,Categorical label for NO2 AQI


In [34]:
administrative_metadata = {
    "license": "CC0: Public Domain",
    "version": "1.0",
    "source_url": "https://www.kaggle.com/datasets/hasibalmuzdadid/global-air-pollution-dataset",
    "access": "Publicly available on Kaggle",
    "provenance_note": "Dataset compiled from publicly available government and OpenAQ sources.",
}

display(pd.DataFrame(
    {"Field": list(administrative_metadata.keys()),
     "Value": [str(v) for v in administrative_metadata.values()]}
))

,Field,Value
0,license,CC0: Public Domain
1,version,1.0
2,source_url,https://www.kaggle.com/datasets/hasibalmuzdadi...
3,access,Publicly available on Kaggle
4,provenance_note,Dataset compiled from publicly available gover...


## Step 3 — Create DataCite Metadata (XML)

DataCite is the standard used by research data repositories for persistent, citable metadata.  
We use the **DataCite Metadata Schema 4.x

Key fields we include:
| Field | Value |
|---|---|
| Identifier | DOI (placeholder) |
| Creator | Dataset author |
| Title | Dataset name |
| Publisher | Kaggle |
| Publication Year | 2022 |
| Resource Type | Dataset |
| Description | Abstract |
| Subjects | Keywords |
| Rights | License |
| Related Identifier | Kaggle URL |

In [35]:
def build_datacite_xml() -> str:
    """
    Builds a DataCite-compliant XML string for the Global Air Pollution Dataset.
    Schema: https://schema.datacite.org/meta/kernel-4/
    """
    NS = "http://datacite.org/schema/kernel-4"
    XSI = "http://www.w3.org/2001/XMLSchema-instance"
    SCHEMA_LOC = (
        "http://datacite.org/schema/kernel-4 "
        "https://schema.datacite.org/meta/kernel-4/metadata.xsd"
    )

    root = ET.Element("resource", attrib={
        "xmlns": NS,
        "xmlns:xsi": XSI,
        "xsi:schemaLocation": SCHEMA_LOC,
    })

    # Identifier
    ident = ET.SubElement(root, "identifier", attrib={"identifierType": "DOI"})
    ident.text = "10.34740/kaggle/dsv/3648132"  # Kaggle-style placeholder DOI

    # Creators
    creators = ET.SubElement(root, "creators")
    creator = ET.SubElement(creators, "creator")
    creator_name = ET.SubElement(creator, "creatorName", attrib={"nameType": "Personal"})
    creator_name.text = "Al Muzdadid, Hasib"
    given = ET.SubElement(creator, "givenName")
    given.text = "Hasib"
    family = ET.SubElement(creator, "familyName")
    family.text = "Al Muzdadid"

    # Titles
    titles = ET.SubElement(root, "titles")
    title = ET.SubElement(titles, "title")
    title.text = "Global Air Pollution Dataset"

    # Publisher
    publisher = ET.SubElement(root, "publisher")
    publisher.text = "Kaggle"

    # Publication Year
    pub_year = ET.SubElement(root, "publicationYear")
    pub_year.text = "2022"

    # Resource Type
    res_type = ET.SubElement(root, "resourceType", attrib={"resourceTypeGeneral": "Dataset"})
    res_type.text = "Dataset"

    # Subjects / Keywords
    subjects = ET.SubElement(root, "subjects")
    for kw in ["air quality", "AQI", "air pollution", "CO", "Ozone", "NO2", "PM2.5", "environment", "global"]:
        subj = ET.SubElement(subjects, "subject")
        subj.text = kw

    # Descriptions
    descriptions = ET.SubElement(root, "descriptions")
    desc = ET.SubElement(descriptions, "description", attrib={"descriptionType": "Abstract"})
    desc.text = (
        "A global dataset containing Air Quality Index (AQI) values and concentration "
        "measurements for major air pollutants — CO, Ozone, NO2, and PM2.5 — across "
        "cities and countries worldwide. Each row represents one city-level observation "
        "with the overall AQI value, categorical AQI label, and sub-index values for "
        "each pollutant. Useful for environmental analysis, pollution comparison, and "
        "data management teaching exercises."
    )

    # Rights (License)
    rights_list = ET.SubElement(root, "rightsList")
    rights = ET.SubElement(rights_list, "rights", attrib={
        "rightsURI": "https://creativecommons.org/publicdomain/zero/1.0/",
        "rightsIdentifier": "CC0-1.0"
    })
    rights.text = "CC0 1.0 Universal (Public Domain Dedication)"

    # Related Identifiers
    related = ET.SubElement(root, "relatedIdentifiers")
    rel_id = ET.SubElement(related, "relatedIdentifier", attrib={
        "relatedIdentifierType": "URL",
        "relationType": "IsSupplementTo"
    })
    rel_id.text = "https://www.kaggle.com/datasets/hasibalmuzdadid/global-air-pollution-dataset"

    # Pretty-print
    raw_xml = ET.tostring(root, encoding="unicode", xml_declaration=False)
    dom = xml.dom.minidom.parseString(raw_xml)
    return '<?xml version="1.0" encoding="UTF-8"?>\n' + dom.toprettyxml(indent="  ")[dom.toprettyxml(indent="  ").index("\n")+1:]


datacite_xml = build_datacite_xml()
print(datacite_xml)

<?xml version="1.0" encoding="UTF-8"?>
<resource xmlns="http://datacite.org/schema/kernel-4" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://datacite.org/schema/kernel-4 https://schema.datacite.org/meta/kernel-4/metadata.xsd">
  <identifier identifierType="DOI">10.34740/kaggle/dsv/3648132</identifier>
  <creators>
    <creator>
      <creatorName nameType="Personal">Al Muzdadid, Hasib</creatorName>
      <givenName>Hasib</givenName>
      <familyName>Al Muzdadid</familyName>
    </creator>
  </creators>
  <titles>
    <title>Global Air Pollution Dataset</title>
  </titles>
  <publisher>Kaggle</publisher>
  <publicationYear>2022</publicationYear>
  <resourceType resourceTypeGeneral="Dataset">Dataset</resourceType>
  <subjects>
    <subject>air quality</subject>
    <subject>AQI</subject>
    <subject>air pollution</subject>
    <subject>CO</subject>
    <subject>Ozone</subject>
    <subject>NO2</subject>
    <subject>PM2.5</subject>
    <subject>environm

In [36]:
# Save DataCite XML to file
xml_output_path = Path("global_air_pollution_datacite.xml")
xml_output_path.write_text(datacite_xml, encoding="utf-8")
print(f"DataCite XML saved to: {xml_output_path.resolve()}")

DataCite XML saved to: C:\Users\Vinay\OneDrive - SRH\SRH\Data Mangement\Individual Assignment\Week 1\Exercise_1_Metadata_Creation\global_air_pollution_datacite.xml


## Step 4 — Parse and Validate the DataCite XML

I parse the saved XML back into Python and display it as a clean table to verify correctness.

In [37]:
def local_name(tag: str) -> str:
    """Strip XML namespace from a tag, e.g. {http://...}title -> title"""
    return tag.split("}")[1] if "}" in tag else tag


def parse_datacite_xml(xml_file: Path) -> dict:
    root = ET.parse(xml_file).getroot()

    identifier = None
    creators = []
    title = None
    publisher = None
    publication_year = None
    resource_type = None
    description = None
    subjects = []
    rights = None

    for el in root.iter():
        name = local_name(el.tag)
        text = (el.text or "").strip()

        if name == "identifier" and text and identifier is None:
            identifier = text
        elif name == "creatorName" and text:
            creators.append(text)
        elif name == "title" and text and title is None:
            title = text
        elif name == "publisher" and text and publisher is None:
            publisher = text
        elif name == "publicationYear" and text and publication_year is None:
            publication_year = text
        elif name == "resourceType" and text and resource_type is None:
            resource_type = text
        elif name == "description" and text and description is None:
            description = text
        elif name == "subject" and text:
            subjects.append(text)
        elif name == "rights" and text and rights is None:
            rights = text

    return {
        "identifier": identifier,
        "creators": ", ".join(creators),
        "title": title,
        "publisher": publisher,
        "publicationYear": publication_year,
        "resourceType": resource_type,
        "subjects": ", ".join(subjects),
        "rights": rights,
        "description": (description[:120] + "...") if description and len(description) > 120 else description,
    }


parsed = parse_datacite_xml(xml_output_path)

table = pd.DataFrame(
    {"Field": list(parsed.keys()), "Value": list(parsed.values())}
)
display(table)

,Field,Value
0,identifier,10.34740/kaggle/dsv/3648132
1,creators,"Al Muzdadid, Hasib"
2,title,Global Air Pollution Dataset
3,publisher,Kaggle
4,publicationYear,2022
5,resourceType,Dataset
6,subjects,"air quality, AQI, air pollution, CO, Ozone, NO..."
7,rights,CC0 1.0 Universal (Public Domain Dedication)
8,description,A global dataset containing Air Quality Index ...


## Step 5 — Create schema.org Metadata (JSON-LD)

schema.org uses JSON-LD format and is understood by search engines (Google Dataset Search).  
We use `@type: Dataset` as required by the assignment.
Key fields:
| JSON-LD Key | Meaning |
|---|---|
| `@context` | Declares the schema.org vocabulary |
| `@type` | Must be `Dataset` |
| `name` | Title |
| `description` | Abstract |
| `creator` | Author |
| `publisher` | Where it was published |
| `identifier` | DOI or URL |
| `keywords` | Searchable terms |
| `license` | Usage rights |
| `distribution` | Downloadable file info |

In [38]:
schemaorg_metadata = {
    "@context": "https://schema.org",
    "@type": "Dataset",
    "name": "Global Air Pollution Dataset",
    "description": (
        "A global dataset containing Air Quality Index (AQI) values and concentration "
        "measurements for major air pollutants — CO, Ozone, NO2, and PM2.5 — across "
        "cities and countries worldwide. Each row represents one city-level observation "
        "with the overall AQI value, categorical AQI label, and sub-index values for "
        "each pollutant. Useful for environmental analysis, pollution comparison, and "
        "data management teaching exercises."
    ),
    "creator": {
        "@type": "Person",
        "name": "Hasib Al Muzdadid",
        "url": "https://www.kaggle.com/hasibalmuzdadid"
    },
    "publisher": {
        "@type": "Organization",
        "name": "Kaggle",
        "url": "https://www.kaggle.com"
    },
    "identifier": "10.34740/kaggle/dsv/3648132",
    "url": "https://www.kaggle.com/datasets/hasibalmuzdadid/global-air-pollution-dataset",
    "datePublished": "2022",
    "keywords": [
        "air quality", "AQI", "air pollution", "CO", "Ozone",
        "NO2", "PM2.5", "environment", "global"
    ],
    "license": "https://creativecommons.org/publicdomain/zero/1.0/",
    "inLanguage": "en",
    "spatialCoverage": "Global",
    "variableMeasured": [
        {"@type": "PropertyValue", "name": "AQI Value", "description": "Overall Air Quality Index (dimensionless)"},
        {"@type": "PropertyValue", "name": "CO AQI Value", "description": "AQI sub-index for Carbon Monoxide"},
        {"@type": "PropertyValue", "name": "Ozone AQI Value", "description": "AQI sub-index for Ozone"},
        {"@type": "PropertyValue", "name": "NO2 AQI Value", "description": "AQI sub-index for Nitrogen Dioxide"},
        {"@type": "PropertyValue", "name": "PM2.5 AQI Value", "description": "AQI sub-index for fine particulate matter PM2.5"},
    ],
    "distribution": {
        "@type": "DataDownload",
        "encodingFormat": "text/csv",
        "contentUrl": "https://www.kaggle.com/datasets/hasibalmuzdadid/global-air-pollution-dataset",
        "name": "global air pollution dataset.csv"
    }
}

# Pretty print
print(json.dumps(schemaorg_metadata, indent=2))

{
  "@context": "https://schema.org",
  "@type": "Dataset",
  "name": "Global Air Pollution Dataset",
  "description": "A global dataset containing Air Quality Index (AQI) values and concentration measurements for major air pollutants \u2014 CO, Ozone, NO2, and PM2.5 \u2014 across cities and countries worldwide. Each row represents one city-level observation with the overall AQI value, categorical AQI label, and sub-index values for each pollutant. Useful for environmental analysis, pollution comparison, and data management teaching exercises.",
  "creator": {
    "@type": "Person",
    "name": "Hasib Al Muzdadid",
    "url": "https://www.kaggle.com/hasibalmuzdadid"
  },
  "publisher": {
    "@type": "Organization",
    "name": "Kaggle",
    "url": "https://www.kaggle.com"
  },
  "identifier": "10.34740/kaggle/dsv/3648132",
  "url": "https://www.kaggle.com/datasets/hasibalmuzdadid/global-air-pollution-dataset",
  "datePublished": "2022",
  "keywords": [
    "air quality",
    "AQI",
  

In [39]:
# Save schema.org JSON-LD to file
json_output_path = Path("global_air_pollution_schemaorg.jsonld")
json_output_path.write_text(
    json.dumps(schemaorg_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print(f"schema.org JSON-LD saved to: {json_output_path.resolve()}")

schema.org JSON-LD saved to: C:\Users\Vinay\OneDrive - SRH\SRH\Data Mangement\Individual Assignment\Week 1\Exercise_1_Metadata_Creation\global_air_pollution_schemaorg.jsonld


## Step 6 — Parse and Validate the schema.org JSON-LD

Reload the saved JSON-LD and extract the key fields into a readable table.

In [40]:
# Re-load from disk (simulates what a parser / search engine would do)
json_output_path = Path("global_air_pollution_schemaorg.jsonld")
with open(json_output_path, encoding="utf-8") as f:
    obj = json.load(f)

print(f"Python object type: {type(obj)}")
print(f"Top-level keys: {list(obj.keys())}")

Python object type: <class 'dict'>
Top-level keys: ['@context', '@type', 'name', 'description', 'creator', 'publisher', 'identifier', 'url', 'datePublished', 'keywords', 'license', 'inLanguage', 'spatialCoverage', 'variableMeasured', 'distribution']


In [41]:
def extract_schemaorg_fields(obj: dict) -> dict:
    """Extract the most important schema.org Dataset fields into a flat dictionary."""
    creator = obj.get("creator", {})
    publisher = obj.get("publisher", {})
    dist = obj.get("distribution", {})
    variables = obj.get("variableMeasured", [])

    return {
        "@type": obj.get("@type"),
        "name": obj.get("name"),
        "identifier": obj.get("identifier"),
        "datePublished": obj.get("datePublished"),
        "creator": creator.get("name") if isinstance(creator, dict) else str(creator),
        "publisher": publisher.get("name") if isinstance(publisher, dict) else str(publisher),
        "license": obj.get("license"),
        "keywords": ", ".join(obj.get("keywords", [])),
        "spatialCoverage": obj.get("spatialCoverage"),
        "distribution format": dist.get("encodingFormat") if isinstance(dist, dict) else None,
        "variables measured": ", ".join(v["name"] for v in variables if "name" in v),
        "description (preview)": (obj.get("description", "")[:120] + "..."),
    }


extracted = extract_schemaorg_fields(obj)
display(pd.DataFrame(
    {"Field": list(extracted.keys()), "Value": list(extracted.values())}
))

,Field,Value
0,@type,Dataset
1,name,Global Air Pollution Dataset
2,identifier,10.34740/kaggle/dsv/3648132
3,datePublished,2022
4,creator,Hasib Al Muzdadid
5,publisher,Kaggle
6,license,https://creativecommons.org/publicdomain/zero/...
7,keywords,"air quality, AQI, air pollution, CO, Ozone, NO..."
8,spatialCoverage,Global
9,distribution format,text/csv


## Step 7 — FAIR Principles Reflection

How does our metadata support the **FAIR** principles?

In [42]:
fair_mapping = [
    {
        "FAIR Principle": "Findable",
        "How our metadata helps": "Title, keywords, description, and DOI identifier allow discovery via search engines and data repositories.",
        "Relevant fields": "name/title, keywords, identifier (DOI)"
    },
    {
        "FAIR Principle": "Accessible",
        "How our metadata helps": "The 'distribution' field provides a direct URL. License CC0 makes access unrestricted.",
        "Relevant fields": "distribution.contentUrl, license, url"
    },
    {
        "FAIR Principle": "Interoperable",
        "How our metadata helps": "DataCite XML and schema.org JSON-LD are internationally recognised, machine-readable standards.",
        "Relevant fields": "@context (schema.org), DataCite schema namespace"
    },
    {
        "FAIR Principle": "Reusable",
        "How our metadata helps": "Column meanings, units, provenance, version, and CC0 license enable confident reuse.",
        "Relevant fields": "variableMeasured, rights/license, creator, datePublished"
    },
]

display(pd.DataFrame(fair_mapping))

,FAIR Principle,How our metadata helps,Relevant fields
0,Findable,"Title, keywords, description, and DOI identifi...","name/title, keywords, identifier (DOI)"
1,Accessible,The 'distribution' field provides a direct URL...,"distribution.contentUrl, license, url"
2,Interoperable,DataCite XML and schema.org JSON-LD are intern...,"@context (schema.org), DataCite schema namespace"
3,Reusable,"Column meanings, units, provenance, version, a...","variableMeasured, rights/license, creator, dat..."


## Step 8 — Summary

In this exercise we:

1. **Selected** a publicly available dataset: *Global Air Pollution Dataset* from Kaggle  
2. **Explored** the dataset structure (columns, data types, statistics)  
3. **Defined** descriptive, structural, and administrative metadata in Python  
4. **Created** a valid **DataCite XML** file (`global_air_pollution_datacite.xml`)  
5. **Created** a valid **schema.org JSON-LD** file (`global_air_pollution_schemaorg.jsonldld`)  
6. **Parsed and validated** both files programmatically  
7. **Mapped** the metadata to the four **FAIR principles**  

### Files produced by this notebook
| File | Format | Standard |
|---|---|---|
| `global_air_pollution_datacite.xml` | XML | DataCite Metadata Schema 4.x |
| `global_air_pollution_schemaorg.jsonldld` | JSON-LD | schema.org (@type: Dataset) |

### Key takeaway
Good metadata makes a dataset **Findable, Accessible, Interoperable, and Reusable (FAIR)**.  
Without metadata, even a well-structured CSV cannot be discovered, trusted, or reused effectively.